
**GE234 Basic Programming**

⛽ **EnergyRoute Optimizer**
**วิเคราะห์เส้นทางและคำนวณการเติมพลังงาน จากกรุงเทพฯ ไป เชียงใหม่**

  ในชีวิตประจำวันการเดินทางไม่ได้ต้องการแค่เส้นทางที่มีระยะทางสั้นที่สุด แต่ต้องการเส้นทางที่สะดวกที่สุด
เช่น การหาสถานีบริการน้ำมันระหว่างทางให้สะดวกและทันท่วงทีที่สุด

เส้นทางจากกรุงเทพไปเชียงใหม่ เป็นเส้นทางที่ได้รับความนิยม แต่ต้องใช้เวลานานและผู้เดินทางต้องวางแผนการแวะสถานีบริการน้ำมันระหว่างการเดินทาง
     โปรเจกต์นี้ จึงออกแบบระบบนำทางที่ไม่ได้บอกแค่เส้นทางที่มีระยะทางสั้นที่สุด แต่ช่วยวิเคราะห์และแนะนำสถานีบริการน้ำมันที่เหมาะสมบนเส้นทางนี้ โดยใช้ข้อมูลจาก OpenStreetMap เพื่อให้การเดินทางสะดวกและมีประสิทธิภาพ

```
วัตถุประสงค์

1.เพื่อออกแบบระบบที่สามารถแนะนำสถานีบริการน้ำมันที่เหมาะสมระหว่างเส้นทางการเดินทาง

2.เพื่อเพิ่มประสิทธิภาพในการวางแผนการเดินทาง  โดยคำนึงถึงระยะทางและจุดบริการระหว่างทาง

3.เพื่อแสดงผลเส้นทางและจุดแวะพักในรูปแบบที่เข้าใจง่ายและใช้งานได้จริง
```



```
ขั้นตอนการทำงาน
1.กำหนดเส้นทางหลัก
2.ดึงข้อมูลปั๊มจาก OSM
3.แปลงข้อมูลเป็นไฟล์ cvs
4.สร้างเว็บ
```


```
# กำหนดเส้นทางหลัก
เลือกเส้นทาง
กรุงเทพฯไปรังสิต (ทางหลวงหมายเลข 1 ถนนพหลโยธิน)
อยุธยา – อ่างทอง – สิงห์บุรี (ทางหลวงหมายเลข 32 ถนนสายเอเชีย)
อ.มโนรมย์ (ทางหลวงหมายเลข 1 ถนนพหลโยธิน)
จากนั้นมุ่งหน้าสู่นครสวรรค์ - เชียงใหม่

โดยเส้นทางได้ผ่านแต่ละจังหวัด ดังนี้
กรุงเทพฯและปริมณฑล, อยุธยา, อ่างทอง, สิงห์บุรี, ชัยนาท, นครสวรรค์, กำแพงเพชร, ตาก, ลำปาง, ลำพูน และเชียงใหม่

```





In [ ]:
!pip install flask flask-cors pyngrok requests

from flask import Flask, jsonify, request
from flask_cors import CORS
import requests
import threading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# 1. ติดตั้ง Library (รันครั้งเดียว)
# ==========================================
!pip -q install flask flask-cors requests pandas

# ==========================================
# 2. Backend (Python + Flask)
# ==========================================
from flask import Flask, jsonify, request
from flask_cors import CORS
import pandas as pd
import requests
from google.colab.output import eval_js

app = Flask(__name__)
CORS(app)

# โหลดข้อมูลจาก CSV
try:
    df_stations = pd.read_csv('/content/drive/MyDrive/all_gas_stations.csv')
    # ล้างข้อมูล NaN ในคอลัมน์น้ำมันเพื่อให้แสดงผลสวยงาม
    df_stations['Available_Fuel_Types'] = df_stations['Available_Fuel_Types'].fillna('ไม่ระบุประเภทน้ำมัน')
    print("✅ เชื่อมต่อไฟล์ gas_stations.csv สำเร็จ!")
except Exception as e:
    df_stations = pd.DataFrame(columns=['Latitude', 'Longitude', 'name', 'Available_Fuel_Types'])
    print(f"❌ เกิดข้อผิดพลาด: {e} (อย่าลืมอัปโหลดไฟล์ที่แถบซ้ายมือ)")

OSRM_URL = "http://router.project-osrm.org/route/v1/driving"

@app.route("/api/route")
def route():
    start = request.args.get("start")
    end = request.args.get("end")
    url = f"{OSRM_URL}/{start};{end}?overview=full&geometries=geojson"
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        return jsonify({
            "geometry": data["routes"][0]["geometry"],
            "distance": data["routes"][0]["distance"]
        })
    except:
        return jsonify({"geometry": None})

@app.route("/api/stations")
def stations():
    bbox = request.args.get("bbox")
    if not bbox or df_stations.empty: return jsonify([])

    minLat, minLon, maxLat, maxLon = map(float, bbox.split(","))

    # กรองปั๊มที่อยู่ในกรอบพื้นที่เส้นทาง
    local_stations = df_stations[
        (df_stations['Latitude'] >= minLat) & (df_stations['Latitude'] <= maxLat) &
        (df_stations['Longitude'] >= minLon) & (df_stations['Longitude'] <= maxLon)
    ]

    out = []
    for _, row in local_stations.iterrows():
        out.append({
            "lat": row['Latitude'],
            "lon": row['Longitude'],
            "brand": row.get('name', 'ปั๊มน้ำมัน'),
            "fuels": row.get('Available_Fuel_Types', 'ไม่ระบุประเภทน้ำมัน')
        })
    return jsonify(out)

# ==========================================
# 3. Frontend (HTML/JS + Leaflet)
# ==========================================
@app.route("/")
def home():
    return """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8"/>
    <title>Fuel Finder & Trip Planner</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
    <style>
        body { margin:0; font-family: 'Tahoma', sans-serif; background: #f4f7f6; }
        .header {
            padding: 15px; background: #1a2a6c; color: white;
            display: flex; align-items: center; justify-content: space-between;
            box-shadow: 0 2px 10px rgba(0,0,0,0.3);
        }
        #map { height: calc(100vh - 85px); width: 100%; }
        .panel { background: white; padding: 10px 20px; border-radius: 8px; color: #333; min-width: 300px; }
        .fuel-badge { color: white; padding: 2px 8px; border-radius: 4px; font-size: 12px; font-weight: bold; }
        .fuel-info { font-size: 0.9em; color: #555; margin-top: 5px; border-top: 1px solid #eee; padding-top: 5px; }
        .alert { color: #ff4757; font-weight: bold; animation: blink 1s infinite; }
        @keyframes blink { 0% { opacity: 1; } 50% { opacity: 0.5; } 100% { opacity: 1; } }
    </style>
</head>
<body>

<div class="header">
    <div style="display:flex; gap:20px; align-items:center;">
        <h2 style="margin:0;">⛽ Trip Planner</h2>
        <div class="panel">
            <strong>น้ำมันเหลือ:</strong>
            <input type="range" id="fuelSlider" min="0" max="100" value="50" oninput="updateFuel(this.value)">
            <span id="fuelLabel" style="font-weight:bold; color:#1a2a6c;">50</span> %
        </div>
    </div>
    <div id="status" style="text-align:right; font-size: 14px;">
        กรุณาคลิกเลือก <b>จุดเริ่ม</b> และ <b>จุดหมาย</b> บนแผนที่
    </div>
</div>

<div id="map"></div>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
    const map = L.map('map').setView([13.75, 100.5], 6);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png').addTo(map);

    let start=null, end=null, routeLine=null;
    let stationLayer = L.layerGroup().addTo(map);
    let fuelPercent = 50;
    const KM_PER_PERCENT = 5;

    function updateFuel(val) {
        fuelPercent = val;
        document.getElementById('fuelLabel').innerText = val;
    }

    // คำนวณระยะพิกัด (Haversine Formula แบบง่าย)
    function getDistance(lon1, lat1, lon2, lat2) {
        const R = 6371;
        const dLat = (lat2-lat1) * Math.PI / 180;
        const dLon = (lon2-lon1) * Math.PI / 180;
        const a = Math.sin(dLat/2) * Math.sin(dLat/2) +
                  Math.cos(lat1 * Math.PI / 180) * Math.cos(lat2 * Math.PI / 180) *
                  Math.sin(dLon/2) * Math.sin(dLon/2);
        const c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
        return R * c;
    }

    function checkNearRoute(lat, lon, coords) {
        let minD = Infinity;
        for(let i=0; i<coords.length; i+=5){
            let d = getDistance(lon, lat, coords[i][0], coords[i][1]);
            if(d < minD) minD = d;
            if(minD < 2) break;
        }
        return minD;
    }

    map.on("click", async e => {
        if(!start){
            start = e.latlng;
            L.marker(start).bindPopup("<b>จุดเริ่มต้น</b>").addTo(map);
            return;
        }
        if(end) { location.reload(); return; }

        end = e.latlng;
        L.marker(end, {icon: L.icon({
            iconUrl: 'https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-red.png',
            iconSize: [25, 41], iconAnchor: [12, 41]
        })}).bindPopup("<b>จุดหมาย</b>").addTo(map);

        const r = await fetch(`/api/route?start=${start.lng},${start.lat}&end=${end.lng},${end.lat}`);
        const data = await r.json();

        if(data.geometry){
            routeLine = L.geoJSON(data.geometry, {style:{color:'#1e90ff', weight:6}}).addTo(map);
            const coords = data.geometry.coordinates;
            const totalKm = data.distance / 1000;
            const canGoKm = fuelPercent * KM_PER_PERCENT;

            let warningText = totalKm > canGoKm ?
                `<br><span class="alert">⚠️ น้ำมันไม่พอนะบะบี๋! แวะเติมด้วยคร้าบบ ขาดอีก ${(totalKm-canGoKm).toFixed(1)} กม.</span>` :
                `<br><span style="color:#2ed573">✅ น้ำมันพอคร้าบบ เดินทางปลอดภัยนะบะบี๋</span>`;

            document.getElementById('status').innerHTML = `ระยะทางรวม: <b>${totalKm.toFixed(1)} กม.</b> | วิ่งไหว: <b>${canGoKm} กม.</b>${warningText}`;

            const res = await fetch(`/api/stations?bbox=${Math.min(start.lat, end.lat)-1},${Math.min(start.lng, end.lng)-1},${Math.max(start.lat, end.lat)+1},${Math.max(start.lng, end.lng)+1}`);
            const stations = await res.json();

            stationLayer.clearLayers();
            stations.forEach(st => {
                const distToRoad = checkNearRoute(st.lat, st.lon, coords);
                if(distToRoad < 4) {
                    const distFromStart = getDistance(start.lng, start.lat, st.lon, st.lat);

                    // --- ตรรกะเลือกสี ---
                    // ถ้าปั๊มอยู่ในระยะที่น้ำมัน "ใกล้จะหมด" (ห่างจากจุดเริ่ม > ระยะที่วิ่งไหว - 30 กม.) ให้เป็นสีแดง
                    let color = "#2ed573"; // เขียว
                    if (distFromStart > canGoKm - 30) {
                        color = "#ff4757"; // แดง
                    }

                    L.circleMarker([st.lat, st.lon], {
                        radius: 10, fillColor: color, color: "#fff", weight: 2, fillOpacity: 0.9
                    }).bindPopup(`
                        <div style="font-family: sans-serif;">
                            <strong>${st.brand}</strong><br>
                            <span class="fuel-badge" style="background:${color}">ห่างจากจุดเริ่ม ${distFromStart.toFixed(1)} กม.</span><br>
                            <small>ห่างจากถนนหลัก: ${distToRoad.toFixed(2)} กม.</small>
                            <div class="fuel-info">
                                <b>⛽ น้ำมันที่จำหน่าย:</b><br>${st.fuels}
                            </div>
                        </div>
                    `).addTo(stationLayer);
                }
            });
        }
    });
</script>
</body>
</html>
"""

# ==========================================
# 4. รันระบบ
# ==========================================
print("--- เริ่มการทำงานของระบบ Trip Planner ---")
print("คลิกที่ลิงก์ด้านล่างเพื่อเปิดหน้าเว็บ:")
print(eval_js("google.colab.kernel.proxyPort(5001)"))
app.run(host="0.0.0.0", port=5001)

✅ เชื่อมต่อไฟล์ gas_stations.csv สำเร็จ!
--- เริ่มการทำงานของระบบ Trip Planner ---
คลิกที่ลิงก์ด้านล่างเพื่อเปิดหน้าเว็บ:
https://5001-m-s-kkb-use1b0-36vnfqtlvp8fq-b.us-east1-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5001)"))

https://5001-m-s-kkb-use1d0-1lsvn514l7q25-d.us-east1-0.prod.colab.dev


ผู้จัดทำ

จิตราภรณ์ วรรณวิลัย

ชลิดา ทิตศานติกุล

จิรัชยา โชคกำเนิด

นรีกานต์ นึกสม